# CMPS 355 — Production Corrected Nested Elastic Net v2 (Fail-Closed & SHA-Guarded)

**Folder:** `peer_review_revision_2026/03_followup_experiments/corrected_nested_elastic_net_20261122_v1/`
**Reviewed Script:** `corrected_nested_elastic_net_v2.py`  
**Expected SHA-256:** `bc36f7379399e73b70e885d153d194b4a10943c04e9c6d1761c482afa71d0925`

- **Cell 1:** Mount Drive & verify R + `glmnet`.
- **Cell 2 (Live R Smoke Test ONLY):** Verifies exact script SHA-256, then runs `--backend r-glmnet --smoke-test` on canonical `D3` (`n=76`) **and** the larger LOCO `D3`-held-out stress partition (`n=613`) with validated checkpoint/resume into a uniquely timestamped `smoke_test_run_<UTC>/` folder.
- **Cell 3 (GATED Full 200-Fit Benchmark):** Verifies exact script SHA-256; guarded by `ALLOW_FULL_200_FIT_RUN = False` until ChatGPT signs off on `v2` code review + live smoke test.
- **Cell 4 (Isolated Finalize-Only Recovery):** Regenerates `manifest.json`, `numerical_warnings.json`, and `SHA256SUMS.txt` with **zero** solver calls while preserving immutable `execution_script_sha256`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, subprocess
if shutil.which('Rscript') is None:
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'r-base-core'], check=True)
chk = subprocess.run(['Rscript', '-e', 'cat(requireNamespace("glmnet", quietly=TRUE))'], capture_output=True, text=True)
if chk.stdout.strip() != 'TRUE':
    apt_res = subprocess.run(['apt-get', 'install', '-y', '-qq', 'r-cran-glmnet'])
    if apt_res.returncode != 0:
        subprocess.run(['Rscript', '-e', 'install.packages("glmnet", repos="https://cloud.r-project.org")'], check=True)
subprocess.run(['Rscript', '-e', 'cat(R.version.string, "\n"); cat("glmnet:", as.character(packageVersion("glmnet")), "\n")'], check=True)

In [ ]:
# CELL 2 — SHA-GUARDED LIVE R SMOKE TEST ONLY (canonical D3 + LOCO D3 n=613 stress fit)
import hashlib, subprocess, sys
from pathlib import Path

EXPECTED_SCRIPT_SHA256 = 'bc36f7379399e73b70e885d153d194b4a10943c04e9c6d1761c482afa71d0925'
BASE = Path('/content/drive/MyDrive/CMPS355_Research')
DATA_ZIP = BASE / 'data/validated/ibd_dataset_sample_aligned.zip'
OUT_DIR = BASE / 'peer_review_revision_2026/03_followup_experiments/corrected_nested_elastic_net_20261122_v1'
SCRIPT = OUT_DIR / 'corrected_nested_elastic_net_v2.py'

assert DATA_ZIP.is_file() and SCRIPT.is_file(), 'Missing canonical ZIP or v2 script'
actual_sha = hashlib.sha256(SCRIPT.read_bytes()).hexdigest()
assert actual_sha == EXPECTED_SCRIPT_SHA256, f'Script SHA mismatch: {actual_sha} != {EXPECTED_SCRIPT_SHA256}'
print('Verified v2 script SHA-256:', actual_sha)

cmd = [
    sys.executable, '-u', str(SCRIPT),
    '--data', str(DATA_ZIP),
    '--out', str(OUT_DIR),
    '--backend', 'r-glmnet',
    '--smoke-test',
]
subprocess.run(cmd, check=True)

In [ ]:
# CELL 3 — FULL 200-FIT CANONICAL BENCHMARK (GATED: keep False until ChatGPT approves v2 review + live smoke test)
ALLOW_FULL_200_FIT_RUN = True
EXPECTED_SCRIPT_SHA256 = 'bc36f7379399e73b70e885d153d194b4a10943c04e9c6d1761c482afa71d0925'

if not ALLOW_FULL_200_FIT_RUN:
    print('Full 200-fit run is GATED (ALLOW_FULL_200_FIT_RUN=False). Awaiting ChatGPT v2 review & live smoke sign-off.')
else:
    import hashlib, subprocess, sys
    from pathlib import Path
    BASE = Path('/content/drive/MyDrive/CMPS355_Research')
    DATA_ZIP = BASE / 'data/validated/ibd_dataset_sample_aligned.zip'
    OUT_DIR = BASE / 'peer_review_revision_2026/03_followup_experiments/corrected_nested_elastic_net_20261122_v1'
    SCRIPT = OUT_DIR / 'corrected_nested_elastic_net_v2.py'
    actual_sha = hashlib.sha256(SCRIPT.read_bytes()).hexdigest()
    assert actual_sha == EXPECTED_SCRIPT_SHA256, f'Script SHA mismatch: {actual_sha} != {EXPECTED_SCRIPT_SHA256}'
    cmd = [
        sys.executable, '-u', str(SCRIPT),
        '--data', str(DATA_ZIP),
        '--out', str(OUT_DIR),
        '--backend', 'r-glmnet',
        '--fit-timeout-sec', '300',
    ]
    log_path = OUT_DIR / 'run.log'
    with log_path.open('a', encoding='utf-8') as lf:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            print(line, end='', flush=True)
            lf.write(line)
            lf.flush()
        rc = proc.wait()
    assert rc == 0, f'Full run exited with code {rc}'

In [ ]:
# CELL 4 — ISOLATED FINALIZE-ONLY RECOVERY (NEVER invokes any solver; preserves immutable execution_script_sha256)
import subprocess, sys
from pathlib import Path
OUT_DIR = Path('/content/drive/MyDrive/CMPS355_Research/peer_review_revision_2026/03_followup_experiments/corrected_nested_elastic_net_20261122_v1')
SCRIPT = OUT_DIR / 'corrected_nested_elastic_net_v2.py'
# Pass --allow-incomplete only if intentionally recovering checksums on a partial run; omit for strict certification
subprocess.run([sys.executable, '-u', str(SCRIPT), '--out', str(OUT_DIR), '--finalize-only'], check=True)